<a href="https://colab.research.google.com/github/Om-merkle/AI_CAP_MED_EMBED/blob/main/notebooks/run_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Om-merkle/AI_CAP_MED_EMBED/blob/main/notebooks/run_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩺 Medical Embedding Fine-Tuning — run on Colab (free T4 GPU)

End-to-end on the **complete, real medical datasets** pulled straight from the Hugging Face Hub:
prepare data → collect (query, positive, negative) triplets → MTEB medical baseline →
fine-tune (**3 epochs**) → evaluate → before/after comparison.

**Setup (one time):** Runtime → **Change runtime type** → Hardware accelerator = **T4 GPU**.

In [1]:
# Cell 0 — confirm the GPU is on (should print a Tesla T4)
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d69e3eff-9987-978a-7c5e-5b736b8b3cb5)


In [2]:
# Cell 1 — clone the project + install dependencies (~2 min)
REPO_URL = "https://github.com/Om-merkle/AI_CAP_MED_EMBED"  # <-- set your repo URL

!git clone {REPO_URL} project
%cd project
!pip install -q -r requirements.txt

Cloning into 'project'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 98 (delta 45), reused 84 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 4.81 MiB | 17.45 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 30.2 MB/s eta 0:00:00


In [3]:
# Cell 2 — optional Hugging Face connection (higher rate limits + enables Hub push).
# Add a secret named HF_TOKEN in Colab's key icon (left sidebar) to use it.
# The datasets are public, so this cell is a safe no-op without a token.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets ✅")
except Exception:
    print("No HF_TOKEN secret found — continuing anonymously (fine for public datasets).")

No HF_TOKEN secret found — continuing anonymously (fine for public datasets).


In [4]:
# Cell 3 — pull the COMPLETE datasets from the Hugging Face Hub (~3-5 min):
#   * all 33,955 medical flashcards (medalpaca)
#   * all ~232,000 MedEmbed clinical triplets (abhinand/MedEmbed-training-triplets-v1)
# This overwrites the small starter sample bundled in data/raw/.
!python scripts/download_dataset.py
!wc -l data/raw/*.jsonl

README.md: 100% 1.24k/1.24k [00:00<00:00, 4.67MB/s]
medical_meadow_wikidoc_medical_flashcard(…): 100% 17.7M/17.7M [00:01<00:00, 16.0MB/s]
Generating train split: 100% 33955/33955 [00:00<00:00, 101767.12 examples/s]
  wrote 33955 rows -> /content/project/data/raw/medical_flashcards.jsonl
README.md: 100% 6.36k/6.36k [00:00<00:00, 17.0MB/s]
data/train-00000-of-00001.parquet: 100% 59.7M/59.7M [00:03<00:00, 15.1MB/s]
Generating train split: 100% 232684/232684 [00:00<00:00, 467839.76 examples/s]
  wrote 232684 rows -> /content/project/data/raw/medembed_triplets.jsonl

Done. The `flashcards` and `medembed` domains now run fully offline.
(`nfcorpus` streams BeIR/nfcorpus from the HF Hub on first use.)
   232684 data/raw/medembed_triplets.jsonl
    33955 data/raw/medical_flashcards.jsonl
   266639 total


## Fine-tune (3 epochs) + evaluate on 4 medical benchmarks

Pick a medical domain:
- `nfcorpus` — BeIR medical/nutrition IR (honest before/after on its own benchmark)
- `medembed` — the complete 232k MedEmbed clinical triplets with expert negatives (mining skipped)
- `flashcards` — 34k medical Q/A flashcards

The pipeline evaluates the **base and fine-tuned models on 4 medical MTEB benchmarks**
(MedicalQARetrieval, PublicHealthQA, NFCorpus, ArguAna) and shows per-benchmark
before/after/delta in the comparison and the leaderboard. Add TRECCOVID (the 5th,
~171k docs, slow) with `--mteb-tasks all`. `--benchmark` additionally compares candidate
models (MiniLM vs BGE vs the published MedEmbed) on your triplets.

Expect roughly 60–90 min on a T4 (training + 4 benchmarks × 2 models); add `--no-mteb`
for a fast training-only run.

In [5]:
!python run_pipeline.py --domain nfcorpus --base-model BAAI/bge-small-en-v1.5 \
    --epochs 3 --batch-size 32 --benchmark --run-label colab-nfcorpus-3ep

[device] cuda  |  base_model=BAAI/bge-small-en-v1.5  domain=nfcorpus  primary_task=NFCorpus
[benchmarks] MedicalQARetrieval, PublicHealthQA, NFCorpus, ArguAna

[1/6] Preparing medical data ...
README.md: 100% 10.5k/10.5k [00:00<00:00, 22.4MB/s]
corpus/corpus-00000-of-00001.parquet: 100% 3.16M/3.16M [00:00<00:00, 3.21MB/s]
Generating corpus split: 100% 3633/3633 [00:00<00:00, 124602.64 examples/s]
queries/queries-00000-of-00001.parquet: 100% 80.9k/80.9k [00:00<00:00, 128kB/s]
Generating queries split: 100% 3237/3237 [00:00<00:00, 898303.70 examples/s]
README.md: 100% 14.0k/14.0k [00:00<00:00, 35.7MB/s]
train.tsv: 100% 2.50M/2.50M [00:00<00:00, 91.4MB/s]
dev.tsv: 100% 258k/258k [00:00<00:00, 183MB/s]
test.tsv: 100% 280k/280k [00:00<00:00, 190MB/s]
Generating train split: 100% 110575/110575 [00:00<00:00, 1267782.23 examples/s]
Generating validation split: 100% 11385/11385 [00:00<00:00, 857494.45 examples/s]
Generating test split: 100% 12334/12334 [00:00<00:00, 943800.66 examples/s]
{
  "d

In [6]:
# Before/after comparison — fine-tuned nDCG@10 should beat the baseline.
import json
print(json.dumps(json.load(open("results/comparison.json")), indent=2))

{
  "base_model": "BAAI/bge-small-en-v1.5",
  "finetuned_model": "/content/project/models/nfcorpus-bge-small-en-v1.5-ft",
  "headline_ir_ndcg@10_delta": 0.0431,
  "improved": true,
  "triplet_accuracy_finetuned": 1.0,
  "rows": [
    {
      "metric": "IR ndcg@10",
      "baseline": 0.3329,
      "finetuned": 0.376,
      "delta": 0.0431
    },
    {
      "metric": "IR mrr@10",
      "baseline": 0.5386,
      "finetuned": 0.6201,
      "delta": 0.0815
    },
    {
      "metric": "IR map@100",
      "baseline": 0.1593,
      "finetuned": 0.185,
      "delta": 0.0257
    },
    {
      "metric": "IR recall@10",
      "baseline": 0.1507,
      "finetuned": 0.1733,
      "delta": 0.0226
    },
    {
      "metric": "MTEB MedicalQARetrieval ndcg@10",
      "baseline": 0.6556,
      "finetuned": 0.627,
      "delta": -0.0286
    },
    {
      "metric": "MTEB PublicHealthQA ndcg@10",
      "baseline": 0.7618,
      "finetuned": 0.7345,
      "delta": -0.0273
    },
    {
      "metric": "M

In [7]:
# Ranked leaderboard of every run so far (persists in results/leaderboard.csv).
from core import leaderboard
leaderboard.to_dataframe()

,run_at,run_label,base_model,domain,device,epochs,batch_size,sample_size,num_triplets,ir_ndcg@10_base,...,mteb_ArguAna_ft,mteb_ArguAna_delta,mteb_TRECCOVID_base,mteb_TRECCOVID_ft,mteb_TRECCOVID_delta,triplet_accuracy,llm_model,llm_input_tokens,llm_output_tokens,llm_cost_usd
0,2026-07-06 05:49:27,colab-nfcorpus-3ep,BAAI/bge-small-en-v1.5,nfcorpus,cuda,3,32,,8238,0.3329,...,0.5511,-0.0439,,,,1.0,,,,


## 🏆 Medical benchmark leaderboard (MedEmbed-style)

The full benchmark table from the MedEmbed project: every model evaluated on the same
official medical MTEB tasks, **nDCG@10 + MRR@5 per task**, best score per column in
**bold**, and your fine-tuned model highlighted in green.

Results are cached in `results/med_benchmarks.json` — re-running only evaluates what's new.
`QUICK_TASKS` skips TRECCOVID (~171k docs, by far the slowest); switch to
`med_leaderboard.DEFAULT_TASKS` for the complete 5-task table (~1-2 h extra on a T4).

In [8]:
from core import med_leaderboard

MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "abhinand/MedEmbed-small-v0.1",
    "models/nfcorpus-bge-small-en-v1.5-ft",   # <- your fine-tuned model (from the run above)
]
TASKS = med_leaderboard.QUICK_TASKS  # or med_leaderboard.DEFAULT_TASKS to add TRECCOVID

med_leaderboard.evaluate(models=MODELS, tasks=TASKS)
med_leaderboard.styled(tasks=TASKS)

[evaluating] sentence-transformers/all-MiniLM-L6-v2 on ['MedicalQARetrieval', 'PublicHealthQA', 'NFCorpus', 'ArguAna']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[evaluating] BAAI/bge-small-en-v1.5 on ['MedicalQARetrieval', 'PublicHealthQA', 'NFCorpus', 'ArguAna']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[evaluating] abhinand/MedEmbed-small-v0.1 on ['MedicalQARetrieval', 'PublicHealthQA', 'NFCorpus', 'ArguAna']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[evaluating] models/nfcorpus-bge-small-en-v1.5-ft on ['MedicalQARetrieval', 'PublicHealthQA', 'NFCorpus', 'ArguAna']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## More experiments (each adds a leaderboard row)

In [9]:
# Train on the COMPLETE MedEmbed clinical triplets (expert negatives, mining skipped).
# ~232k triplets x 3 epochs is a long T4 run — cap with --sample-size if short on time.
!python run_pipeline.py --domain medembed --epochs 3 --batch-size 32 --run-label colab-medembed-3ep

# Or the complete medical flashcards:
# !python run_pipeline.py --domain flashcards --epochs 3 --batch-size 32 --run-label colab-flashcards-3ep

[device] cuda  |  base_model=BAAI/bge-small-en-v1.5  domain=medembed  primary_task=MedicalQARetrieval
[benchmarks] MedicalQARetrieval, PublicHealthQA, NFCorpus, ArguAna

[1/6] Preparing medical data ...
{
  "domain": "medembed",
  "num_pairs": 21585,
  "num_eval_queries": 100,
  "eval_corpus_size": 5000,
  "pairs_path": "/content/project/data/pairs.jsonl",
  "eval_path": "/content/project/data/eval.json"
}

[2/6] Collecting triplets ...
{
  "source": "native (dataset ships expert triplets; mining skipped)",
  "num_triplets": 21585,
  "triplets_path": "/content/project/data/triplets.jsonl"
}

[3/6] MTEB / IR baseline (base model) ...
Loading weights: 100% 199/199 [00:00<00:00, 6401.41it/s]
{
  "model": "BAAI/bge-small-en-v1.5",
  "ir": {
    "ndcg@10": 0.38,
    "mrr@10": 0.351,
    "map@100": 0.3575,
    "recall@10": 0.47,
    "accuracy@1": 0.29
  },
  "mteb": {
    "task": "MedicalQARetrieval",
    "ndcg@10": 0.6556,
    "tasks": {
      "MedicalQARetrieval": {
        "ndcg@10": 0.65

## Download the fine-tuned model + results

In [10]:
!zip -rq finetuned_model.zip models/ results/
from google.colab import files
files.download("finetuned_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Use the model afterwards:**

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("models/nfcorpus-bge-small-en-v1.5-ft")
emb = model.encode(["What are the symptoms of type 2 diabetes?"], normalize_embeddings=True)
```